# Employee Turnover Analytics (Portobello Tech)
This notebook guides you through analyzing employee work details and predicting turnover using Machine Learning.

### Project Objectives:
1. **Data Quality Checks:** Inspect missing values and general dataset stats.
2. **Exploratory Data Analysis (EDA):** Understand linear correlations, variable distributions, and workload-turnover patterns.
3. **K-Means Clustering:** Group departed employees into 3 distinct personas based on satisfaction and evaluations.
4. **Class Imbalance & Preprocessing:** Encode categorical features, split data (80:20 stratified), and upsample training set using SMOTE.
5. **Model Training & Cross-Validation:** Fit Logistic Regression, Random Forest, and Gradient Boosting models using 5-fold CV.
6. **Model Evaluation:** Construct ROC curves, compute AUC, evaluate confusion matrices, and justify metrics selection.
7. **Risk Stratification & Retention:** Categorize employees into 4 risk zones and define targeted retention strategies.


In [ ]:
# Detect environment and setup file imports for Google Colab compatibility
import os
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Google Colab environment.')
    print("Please upload the dataset file 'HR_comma_sep.csv' when prompted below...")
    from google.colab import files
    uploaded = files.upload()
else:
    print('Running in local Python environment. Path defaults to raw data folder.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from imblearn.over_sampling import SMOTE

# Set seaborn theme and plot sizes
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Path resilience: checks local directory path first, then Colab upload folder
try:
    df = pd.read_csv('HR_comma_sep.csv')
except FileNotFoundError:
    try:
        df = pd.read_csv('Project 4.1 Employee Turnover Analytics/HR_comma_sep.csv')
    except FileNotFoundError:
        df = pd.read_csv('/media/jignesh/Data/ihfc/IHFC-Leaning/Project 4.1 Employee Turnover Analytics/HR_comma_sep.csv')

print(f'Dataset loaded successfully. Shape: {df.shape}')


## Step 1: Initial Data Quality Checks & Data Cleaning

### Why do we perform this step?
* **Data Quality Checks** ensure that our model does not fail during execution because of missing/null values.
* We check dimensions, data types, and check for missing values to prepare the data for downstream tasks.


In [ ]:
# Inspect general metadata info
print('=== Metadata Info ===')
df.info()

print('\n=== Missing Values Check ===')
print(df.isna().sum())

# Print descriptive statistics of columns
print('\n=== Summary Statistics (Raw Data) ===')
print(df.describe())


### Step 1 Observations:
* **Shape:** The dataset contains $14,999$ rows (employees) and $10$ columns.
* **Missing Values:** There are no null or missing values across any columns. The dataset is fully complete.
* **Data Types:** The columns are numerical, except for `sales` (department) and `salary` which are categorical strings. Note: `sales` represents the department of the employee.


## Step 2: Exploratory Data Analysis & Feature Correlation

### 2.1 Correlation Heatmap of Numeric Features
To understand linear relationships, we will compute Pearson correlation coefficients ($r$) between the numerical columns, then plot them using a heatmap.


In [ ]:
# Select numerical columns for correlation mapping
numeric_cols = ['satisfaction_level', 'last_evaluation', 'number_project', 
                'average_montly_hours', 'time_spend_company', 'Work_accident', 
                'left', 'promotion_last_5years']
corr_matrix = df[numeric_cols].corr()

# Visualize using Seaborn heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

print('\nCorrelation of Features with left status (Sorted):')
print(corr_matrix['left'].sort_values(ascending=False))


### Step 2.1 Observations:
* **Negative Correlation:** Job satisfaction (`satisfaction_level`) has the strongest negative correlation ($-0.388$) with the target `left`. This indicates that lower satisfaction is the strongest linear indicator of employee turnover.
* **Positive Correlation:** Time spent in the company (`time_spend_company`) ($0.145$) and average monthly working hours ($0.071$) show positive correlations, suggesting that tenure and workload play roles in turnover.


### 2.2 Distribution Plots
We draw kernel density estimation (KDE) distribution plots for Employee Satisfaction (`satisfaction_level`), Employee Evaluation (`last_evaluation`), and Employee Average Monthly Hours (`average_montly_hours`).


In [ ]:
# Draw KDE plots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.kdeplot(data=df, x='satisfaction_level', ax=axes[0], fill=True, color='tab:blue')
axes[0].set_title('Employee Satisfaction Distribution')
axes[0].set_xlabel('Satisfaction Level')

sns.kdeplot(data=df, x='last_evaluation', ax=axes[1], fill=True, color='tab:green')
axes[1].set_title('Employee Evaluation Distribution')
axes[1].set_xlabel('Last Evaluation Rating')

sns.kdeplot(data=df, x='average_montly_hours', ax=axes[2], fill=True, color='tab:red')
axes[2].set_title('Employee Average Monthly Hours Distribution')
axes[2].set_xlabel('Average Monthly Hours')

plt.tight_layout()
plt.show()


### Step 2.2 Observations:
* **Satisfaction Level:** Peaks around extremely low satisfaction ($0.10$), moderate satisfaction ($0.40$), and very high satisfaction ($0.80$).
* **Last Evaluation:** Displays peaks around $0.50$ (average evaluation) and $0.85$ (highly evaluated employees).
* **Average Monthly Hours:** Bimodal distribution peaking at $150$ hours/month (standard/low work hours) and $250$ hours/month (exceptionally long working hours).


### 2.3 Workload Analysis: Project Count vs. Turnover
We draw a count plot of the employee project count, grouped by whether they left or stayed in the company, to investigate how workload impacts turnover.


In [ ]:
# Plot count plot with left as hue
sns.countplot(data=df, x='number_project', hue='left', palette='Set2')
plt.title('Employee Project Count by Left Status')
plt.xlabel('Number of Projects')
plt.ylabel('Count of Employees')
plt.legend(title='Left', labels=['Stayed (0)', 'Left (1)'])
plt.tight_layout()
plt.show()


### Step 2.3 Observations:
* **Underutilization:** Employees with only $2$ projects show high turnover. They may feel disengaged, unmotivated, or are quiet quitting.
* **Optimal Range:** Employees with $3$, $4$, and $5$ projects show high retention, representing balanced and healthy workload zones.
* **Extreme Burnout:** Employees assigned to $6$ or $7$ projects show exceptionally high turnover rates (near $100\%$ for $7$ projects). This is a strong indicator of overworking and burnout.


## Step 3: Cluster Analysis of Departed Employees

We segment employees who left the organization (`left == 1`) into 3 cohorts using **K-Means Clustering** based on `satisfaction_level` and `last_evaluation` to identify distinct departure personas.


In [ ]:
# Filter for departed employees
df_left = df[df['left'] == 1].copy()

# Extract features for clustering
cluster_features = ['satisfaction_level', 'last_evaluation']
X_clust = df_left[cluster_features]

# Run K-Means with K=3
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
df_left['cluster'] = kmeans_model.fit_predict(X_clust)

# Visualize cluster partitions
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df_left['satisfaction_level'], df_left['last_evaluation'], 
                      c=df_left['cluster'], cmap='viridis', alpha=0.6, edgecolors='w', s=50)
plt.legend(*scatter.legend_elements(), title='Personas')
plt.title('K-Means Clustering of Employees Who Left (K=3)')
plt.xlabel('Satisfaction Level')
plt.ylabel('Last Evaluation Rating')
plt.show()

# Display cluster averages
cohort_means = df_left.groupby('cluster')[cluster_features].mean()
print('=== Cohort Average satisfaction and evaluations ===')
print(cohort_means)

print('\nNumber of Departed Employees per Cluster:')
print(df_left['cluster'].value_counts())


### Step 3 observations & Cluster Personas:

Clustering the departed workforce reveals three distinct departure personas:

1. **Cohort: Overworked & Burned Out Stars (Low Satisfaction, High Evaluation)**
   * *Characteristics:* Satisfaction around $0.10$ and evaluations above $0.85$. These represent highly productive and talented contributors who left due to extreme burnout and lack of support.

2. **Cohort: Underutilized & Bored / Underperforming (Moderate Satisfaction, Low Evaluation)**
   * *Characteristics:* Satisfaction around $0.40$ and evaluations around $0.50$. These represent employees who were underperforming or disengaged and left due to poor performance fits or lack of interest.

3. **Cohort: High-Performing Competitive Leavers (High Satisfaction, High Evaluation)**
   * *Characteristics:* Satisfaction around $0.80$ and evaluations above $0.90$. These are stellar employees who loved their jobs and excelled, but were likely headhunted away by competing firms offering better compensation or growth.


## Step 4: Data Preprocessing & Resolving Class Imbalance (SMOTE)

### Why do we preprocess and address class imbalance?
* **Categorical Conversion:** Models require numerical inputs. We convert string columns `sales` (department) and `salary` to numeric columns using one-hot encoding.
* **Class Imbalance:** Only $24\%$ of employees in the dataset left. Standard models trained on this would bias heavily towards predicting that everyone stays. SMOTE upsamples the minority class in the training partition by synthesizing new samples.


In [ ]:
# Apply one-hot encoding to sales and salary variables
df_encoded = pd.get_dummies(df, columns=['sales', 'salary'], drop_first=False)

# Convert Boolean columns generated by get_dummies to integers (0/1)
for col in df_encoded.columns:
    if df_encoded[col].dtype == bool:
        df_encoded[col] = df_encoded[col].astype(int)

# Separate features and target label
X = df_encoded.drop(columns=['left'])
y = df_encoded['left']

# Perform Stratified Train-Test Split (80:20) using random_state=123
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=123
)

print(f'Original Train Set class count:\n{y_train.value_counts()}')

# Perform SMOTE on training set only
smote = SMOTE(random_state=123)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'\nSMOTE Resampled Train Set class count:\n{y_train_res.value_counts()}')


## Step 5: Model Training & 5-Fold Cross-Validation

We train three models using 5-fold cross-validation on the SMOTE-resampled training partition and report their average training metrics.


In [ ]:
# Initialize classifiers
lr_model = LogisticRegression(max_iter=1000, random_state=42)
rf_model = RandomForestClassifier(random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

classifiers = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

# Define 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in classifiers.items():
    print(f'=== Evaluating {name} using 5-Fold CV ===')
    # Generate cross-validated predictions on the training data
    y_cv_pred = cross_val_predict(model, X_train_res, y_train_res, cv=kf)
    print(classification_report(y_train_res, y_cv_pred))


## Step 6: Model Evaluation & Metric Selection

### 6.1 ROC Curves, AUC Scores, and Confusion Matrices on Test Set
We fit the models on the resampled training partition and evaluate their performance on the original test set.


In [ ]:
# Fit models on the SMOTE resampled training set
for name, model in classifiers.items():
    model.fit(X_train_res, y_train_res)

# Plot ROC Curves and compute AUC scores on Test Set
plt.figure(figsize=(10, 8))

for name, model in classifiers.items():
    # Get probabilities for left class (1)
    y_probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    auc_score = roc_auc_score(y_test, y_probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves on Test Dataset')
plt.legend(loc='lower right')
plt.show()

# Construct and display Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(classifiers.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name} Confusion Matrix')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.show()


### 6.2 Evaluation & Metric Justification:

#### A. Best Model Identification:
* **Random Forest Classifier** achieves the highest Area Under the Curve (**AUC = 0.9926**) and the lowest classification errors, closely followed by the Gradient Boosting Classifier (AUC = 0.9885).

#### B. Recall vs. Precision:
* **Recall** is the critical metric for this business case. Recall measures out of all employees who will actually leave, how many we successfully identify.
* **Reasoning:**
  * A **False Negative (FN)** occurs when the model predicts an employee will stay, but they actually leave. This is highly costly: the company loses a valuable resource, incurs recruitment expenses, and suffers lost productivity.
  * A **False Positive (FP)** occurs when the model predicts an employee will leave, but they planned to stay. The cost is minimal: the company may offer them a retention check-in, minor workload adjustments, or small incentives.
* Since the cost of a False Negative is much higher than a False Positive, we prioritize **Recall** to capture as many flight-risk employees as possible.


## Step 7: Risk Stratification & Retention Strategy mapping

We use our best model (Random Forest Classifier) to estimate turnover probabilities ($p$) on the test set and map employees into four distinct risk zones, providing specific retention recommendations.


In [ ]:
# Extract probabilities using Random Forest
test_probs = rf_model.predict_proba(X_test)[:, 1]

# Categorize test set employees into risk zones
risk_zones = []
for prob in test_probs:
    if prob < 0.20:
        risk_zones.append('Safe Zone (Green)')
    elif prob < 0.60:
        risk_zones.append('Low-Risk Zone (Yellow)')
    elif prob < 0.90:
        risk_zones.append('Medium-Risk Zone (Orange)')
    else:
        risk_zones.append('High-Risk Zone (Red)')

# Create results DataFrame
df_results = pd.DataFrame({
    'Actual_Left': y_test,
    'Turnover_Probability': test_probs,
    'Risk_Zone': risk_zones
})

print('=== Risk Zone Counts on Test Set ===')
print(df_results['Risk_Zone'].value_counts())

# Actual turnover counts within each risk zone
zone_stats = df_results.groupby('Risk_Zone')['Actual_Left'].agg(['count', 'sum']).rename(
    columns={'sum': 'Actual_Left_Count', 'count': 'Total_Employees'}
)
zone_stats['Actual_Turnover_Rate'] = (zone_stats['Actual_Left_Count'] / zone_stats['Total_Employees']).round(4)
print('\n=== Risk Zone Performance Stats ===')
print(zone_stats)


### Risk Zone Profiles and Suggested Retention Strategies:

Based on the probability scores, we recommend the following target retention actions:

1. **Safe Zone (Green) (Score < 20%):**
   * *Characteristics:* Highly satisfied and balanced workload. Low flight risk.
   * *Retention Action:* No immediate intervention required. Maintain standard career progression plans, periodic reviews, and positive workplace culture.

2. **Low-Risk Zone (Yellow) (20% <= Score < 60%):**
   * *Characteristics:* Mild indicators of disengagement or slight workload elevation.
   * *Retention Action:* Proactive check-ins by immediate managers. Ensure proper recognition of achievements and access to career growth/training resources.

3. **Medium-Risk Zone (Orange) (60% <= Score < 90%):**
   * *Characteristics:* Moderate disengagement, high hours, or lack of promotion. Clear flight-risk signs.
   * *Retention Action:* Conduct structured 1-on-1 retention discussions. Implement workload redistribution to prevent burnout, review salary/compensation positioning, and explore role/team transfers.

4. **High-Risk Zone (Red) (Score >= 90%):**
   * *Characteristics:* Highly evaluated top performers experiencing severe burnout or extreme dissatisfaction. Immediate flight risk.
   * *Retention Action:* Urgent manager and HR intervention. Immediately reduce workload (remove from low-priority projects), review financial incentives/promotions, offer mandatory rest/mental health days, and re-negotiate roles to align with employee interests.
